## Imports


In [1]:
!bash /kaggle/input/notebooks/packagemanager/pm-111932015-at-03-12-2026-04-24-10/install_requirements.sh

Looking in links: /kaggle/input/notebooks/packagemanager/pm-111932015-at-03-12-2026-04-24-10
Processing /kaggle/input/notebooks/packagemanager/pm-111932015-at-03-12-2026-04-24-10/openvino-2026.0.0-20965-cp312-cp312-manylinux_2_28_x86_64.whl
Processing /kaggle/input/notebooks/packagemanager/pm-111932015-at-03-12-2026-04-24-10/openvino_telemetry-2025.2.0-py3-none-any.whl (from openvino)


In [2]:
import os
import re
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import torchvision
import timm
import openvino as ov
from tqdm import tqdm
import warnings

## Setup


In [3]:
warnings.filterwarnings("ignore")

class Config:
    sr = 32000
    duration = 5
    chunk_len = 32000 * 5
    half_chunk = chunk_len // 2
    n_mels = 256
    n_fft = 2048
    f_min = 20
    f_max = 16000
    model_name = "tf_efficientnetv2_s"
    models_dir = "/kaggle/input/notebooks/lakhindarpal/birdclef-2026-training/models/"
    test_path = "/kaggle/input/competitions/birdclef-2026/test_soundscapes/"
    train_path = "/kaggle/input/competitions/birdclef-2026/train_soundscapes/"

sample_sub = pd.read_csv("/kaggle/input/competitions/birdclef-2026/sample_submission.csv")
primary_labels = sample_sub.columns[1:].tolist()

oggs = glob.glob(Config.test_path + "*.ogg")
if len(oggs) == 0:
    oggs = sorted(glob.glob(Config.train_path + "*.ogg"))[:8]

oggs = [(n, ogg, re.search(r"/([^/]+)\.ogg$", ogg).group(1), True) for n, ogg in enumerate(oggs)]

## Architecture


In [4]:
class SpectrogramPipeline(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=Config.sr, n_fft=Config.n_fft, win_length=Config.n_fft,
            hop_length=512, n_mels=Config.n_mels, f_min=Config.f_min, f_max=Config.f_max,
            power=2.0, norm="slaney", mel_scale="htk"
        )
        self.resize = torchvision.transforms.Resize((256, 256))

    def forward(self, x):
        if x.dim() == 1: x = x.unsqueeze(0)
        mel = self.mel_transform(x)
        amin = 1e-10
        
        log_spec = 10.0 * torch.log10(mel.clamp(min=amin))
        
        max_val = log_spec.flatten(-2).max(dim=-1).values[..., None, None]
        log_spec = log_spec - max_val
        
        log_spec = torch.maximum(log_spec, torch.tensor(-80.0, device=log_spec.device))
        
        return self.resize(log_spec.unsqueeze(1).repeat(1, 3, 1, 1))

class BirdModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(Config.model_name, pretrained=False, num_classes=len(primary_labels), in_chans=3)
    def forward(self, x):
        return self.backbone(x)

## OpenVINO


In [5]:
print("Loading EfficientNet models")
compiled_models = []
core = ov.Core()
ckpt_paths = glob.glob(f"{Config.models_dir}/*.pth")
os.makedirs("/kaggle/working/ov_models", exist_ok=True)

for ckpt in ckpt_paths:
    ov_path = f"/kaggle/working/ov_models/{os.path.basename(ckpt)}.xml"
    if not os.path.exists(ov_path):
        model = BirdModel()
        model.load_state_dict(torch.load(ckpt, map_location="cpu"))
        model.eval()
        ov_model = ov.convert_model(model, example_input=torch.zeros(1, 3, 256, 256))
        ov_model.reshape([-1, 3, 256, 256])
        ov.save_model(ov_model, ov_path)
    
    compiled = core.compile_model(core.read_model(ov_path), device_name="CPU")
    compiled_models.append(compiled)

spec = SpectrogramPipeline()

Loading EfficientNet models


## Inference


In [6]:
print("Running EfficientNet inference")
all_preds = []
all_names = []

for _, filepath, filename, _ in tqdm(oggs):
    try:
        wav, _ = torchaudio.load(filepath)
        wav = wav.float()[0]
        n_seg = len(wav) // Config.chunk_len
        
        if n_seg == 0:
            wav = torch.nn.functional.pad(wav, (0, Config.chunk_len - len(wav)))
            n_seg = 1
        else:
            wav = wav[: n_seg * Config.chunk_len]

        wav_std = wav.reshape((n_seg, Config.chunk_len))
        wav_padded = torch.nn.functional.pad(wav, (0, Config.half_chunk))
        
        wav_shift = torch.stack([
            wav_padded[i * Config.chunk_len + Config.half_chunk : (i + 1) * Config.chunk_len + Config.half_chunk]
            for i in range(n_seg)
        ])

        mel_std = torch.stack([spec(wav_std[i]).squeeze(0) for i in range(n_seg)]).numpy()
        mel_shift = torch.stack([spec(wav_shift[i]).squeeze(0) for i in range(n_seg)]).numpy()
        names = [f"{filename}_{int((i+1)*Config.duration)}" for i in range(n_seg)]

        logits_std = [compiled(mel_std)[compiled.output(0)] for compiled in compiled_models]
        logits_shift = [compiled(mel_shift)[compiled.output(0)] for compiled in compiled_models]

        preds_std = 1 / (1 + np.exp(-np.mean(logits_std, axis=0)))
        preds_shift = 1 / (1 + np.exp(-np.mean(logits_shift, axis=0)))
        
        preds = np.copy(preds_std)
        for i in range(n_seg):
            shift_prev = preds_shift[i - 1] if i > 0 else preds_std[i]
            shift_curr = preds_shift[i]
            preds[i] = 0.50 * preds_std[i] + 0.25 * shift_prev + 0.25 * shift_curr

        smoothed = np.copy(preds)
        for i in range(n_seg):
            p_m2 = preds[max(0, i - 2)]
            p_m1 = preds[max(0, i - 1)]
            p0 = preds[i]
            p_p1 = preds[min(n_seg - 1, i + 1)]
            p_p2 = preds[min(n_seg - 1, i + 2)]
            smoothed[i] = 0.05 * p_m2 + 0.15 * p_m1 + 0.60 * p0 + 0.15 * p_p1 + 0.05 * p_p2

        file_max = np.max(smoothed, axis=0)
        final_preds = 0.80 * smoothed + 0.20 * file_max

        all_preds.append(final_preds)
        all_names.extend(names)

    except:
        n_seg = int(240 / Config.duration)
        all_preds.append(np.zeros((n_seg, len(primary_labels))))
        all_names.extend([f"{filename}_{int((i+1)*Config.duration)}" for i in range(n_seg)])

Running EfficientNet inference


100%|██████████| 8/8 [00:53<00:00,  6.64s/it]


## Submission


In [7]:
final_df = pd.DataFrame(np.concatenate(all_preds, axis=0), columns=primary_labels)
final_df.insert(0, "row_id", all_names)
final_df.to_csv("submission.csv", index=False)
print("Submission saved:", final_df.shape)

Submission saved: (96, 235)


In [8]:
pd.read_csv("/kaggle/working/submission.csv").head()

,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Train_0001_S08_20250606_030007_5,0.003947,0.000247,0.000338,0.000467,0.003839,0.000156,0.000222,0.004178,0.000772,...,0.000170,0.001234,0.000696,0.000119,0.000134,0.000654,0.000163,0.000233,0.000287,0.001643
1,BC2026_Train_0001_S08_20250606_030007_10,0.003943,0.000260,0.000305,0.000483,0.003335,0.000154,0.000242,0.004802,0.000732,...,0.000165,0.001328,0.000733,0.000124,0.000155,0.000671,0.000164,0.000216,0.000316,0.001752
2,BC2026_Train_0001_S08_20250606_030007_15,0.004097,0.000244,0.000312,0.000467,0.002746,0.000169,0.000261,0.005298,0.000661,...,0.000173,0.001256,0.000704,0.000128,0.000166,0.000556,0.000173,0.000216,0.000346,0.001623
3,BC2026_Train_0001_S08_20250606_030007_20,0.004345,0.000229,0.000332,0.000426,0.002512,0.000164,0.000216,0.005431,0.000672,...,0.000165,0.001300,0.000644,0.000126,0.000156,0.000581,0.000179,0.000192,0.000303,0.001625
4,BC2026_Train_0001_S08_20250606_030007_25,0.004505,0.000248,0.000311,0.000445,0.002766,0.000172,0.000210,0.005108,0.000791,...,0.000151,0.001318,0.000626,0.000118,0.000156,0.000682,0.000173,0.000189,0.000277,0.001465
